# Phase 2 — Cross-Validation (the long phase)

**MSc Credit Risk Project — Colab handover (2 of 6)**

Runs the **3x5 repeated stratified cross-validation** over all 9 configurations
(3 model families x 3 imbalance strategies = **135 model fits**):

| Family | Strategies |
|---|---|
| Logistic Regression | Baseline / SMOTE / Cost-Sensitive |
| Random Forest | Baseline / SMOTE / Cost-Sensitive |
| XGBoost | Baseline / SMOTE / Cost-Sensitive |

Hyperparameters are **frozen** across imbalance strategies so the comparison is fair.
Primary metric: **PR-AUC** (accuracy is misleading at 8% base rate).

**Checkpoint:** `reports/tables/cv_fold_results.csv` is rewritten after **each repeat**.
If the runtime disconnects, re-run this notebook — it detects the partial checkpoint and
resumes from the next repeat (seeds are tied to the global repeat number, so results
stay bit-identical).

**Runtime:** ~30-45 minutes on Colab CPU for all 3 repeats.


In [ ]:
# ============================================================
# SETUP: mount Google Drive and locate the project folder
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# AUTO-DETECT the project root (folder containing src/ and run_experiment.py).
# If auto-detection fails, set PROJECT_ROOT manually, e.g.:
#   PROJECT_ROOT = Path('/content/drive/MyDrive/credit-risk-project - Copy')
PROJECT_ROOT = None
for candidate in Path('/content/drive/MyDrive').rglob('run_experiment.py'):
    if (candidate.parent / 'src').is_dir():
        PROJECT_ROOT = candidate.parent
        break
assert PROJECT_ROOT is not None, "Could not find the project folder on Drive - set PROJECT_ROOT manually"

os.chdir(PROJECT_ROOT)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
# Ensure expected directories exist (log files are opened at import time)
for _d in ('logs', 'data/raw', 'data/processed', 'models',
           'reports/tables', 'reports/figures', 'outputs/eda'):
    os.makedirs(PROJECT_ROOT / _d, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print("Working directory set. All outputs are saved here (persistent on Drive).")


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

import pandas as pd
from src.config import TABLES_DIR, N_REPEATS

cv_path = TABLES_DIR / "cv_fold_results.csv"

if cv_path.exists():
    prior = pd.read_csv(cv_path)
    done = int(prior['repeat'].max())
    if done >= N_REPEATS:
        print(f"Full checkpoint found ({done}/{N_REPEATS} repeats) - SKIPPING CV track.")
        df_cv = prior
    else:
        print(f"Partial checkpoint ({done}/{N_REPEATS} repeats) - RESUMING from repeat {done + 1}...")
        from run_experiment import run_cv_track, load_and_split
        df_train, _ = load_and_split()
        df_new = run_cv_track(df_train, repeat_offset=done, checkpoint_path=cv_path)
        df_cv = pd.concat([prior, df_new], ignore_index=True)
        df_cv.to_csv(cv_path, index=False)
else:
    print("No checkpoint - running full CV track (3 repeats x 5 folds x 9 configs = 135 fits)...")
    from run_experiment import run_cv_track, load_and_split
    df_train, _ = load_and_split()
    df_cv = run_cv_track(df_train, checkpoint_path=cv_path)
    df_cv.to_csv(cv_path, index=False)

print(f"CV track complete: {len(df_cv)} fold records (expected 135).")


In [ ]:
# ---- Display the aggregated CV results (Table 4.1 of the dissertation) ----
agg = (df_cv.groupby('display_name')
       .agg(roc_auc_mean=('roc_auc', 'mean'), roc_auc_std=('roc_auc', 'std'),
            pr_auc_mean=('pr_auc', 'mean'), pr_auc_std=('pr_auc', 'std'),
            recall_mean=('recall', 'mean'), precision_mean=('precision', 'mean'),
            f1_mean=('f1', 'mean'), f2_mean=('f2', 'mean'),
            cost_mean=('cost_per_applicant', 'mean'))
       .sort_values('pr_auc_mean', ascending=False))
agg.to_csv(TABLES_DIR / "cv_summary.csv", index=False)
print("Aggregated CV results (sorted by mean PR-AUC):")
agg.round(4)


In [ ]:
# ---- Phase 2 verification against dissertation Table 4.1 ----
xgb_base = agg.loc['XGBoost (Baseline)']
assert abs(xgb_base['pr_auc_mean'] - 0.2502) < 0.005, f"PR-AUC {xgb_base['pr_auc_mean']:.4f} != ~0.2502"
assert abs(xgb_base['roc_auc_mean'] - 0.7622) < 0.005
assert len(df_cv) == 135
print("\nPHASE 2 CHECKPOINT OK: champion XGBoost (Baseline) PR-AUC ~0.2502, ROC-AUC ~0.7622")
print("Note: SMOTE rows should show LOWER PR-AUC than their baselines - SMOTE is harmful here.")
